In [ ]:
# Cell 1: Environment Installation
!pip install xgboost pandas scikit-learn fastapi uvicorn mlflow PyYAML httpx pydantic-settings pytest pytest-asyncio

# Cell 2: Trigger the Pipe-and-Filter Training Phase
from src.train import run_pipe_and_filter_training
run_pipe_and_filter_training()

# Cell 3: Boot ALL 3 Independent Microservice Node Clusters in Parallel
import subprocess
import time

print("Launching Microservice Node 1: Fraud Prediction Cluster [Port 8001]...")
f_proc = subprocess.Popen(["python", "-m", "src.agents.fraud_predictor"])
time.sleep(2)

print("Launching Microservice Node 2: Pipe-and-Filter Compliance RAG [Port 8002]項目...")
r_proc = subprocess.Popen(["python", "-m", "src.agents.compliance_rag"])
time.sleep(2)

print("Launching Microservice Node 3: Central Saga & CQRS Gateway [Port 8000]...")
g_proc = subprocess.Popen(["python", "-m", "src.main"])

print("\n🎉 Everything is fully mapped and running! Send payloads to: http://127.0.0.1:8000/api/v1/adjudicate")

In [ ]:
# Run this for testing
import httpx
import json

url = "http://127.0.0.1:8000/api/v1/adjudicate"

# TEST 1: The Happy Path (Low Risk Claim)
payload_happy = {
    "policy_holder_id": "POL_BITS_001",
    "customer_age": 34,
    "policy_deductible": 1000,
    "claim_amount": 45000.0,
    "past_claims_count": 0,
    "incident_hour": 14
}

# TEST 2: The Early Exit Path (High-Risk Fraud Profile)
payload_fraud = {
    "policy_holder_id": "POL_BITS_002",
    "customer_age": 21,
    "policy_deductible": 500,
    "claim_amount": 140000.0,
    "past_claims_count": 4,
    "incident_hour": 2  # Deep night + high amount triggers early exit
}

with httpx.Client() as client:
    print("--- SENDING HAPPY PATH PAYLOAD ---")
    r1 = client.post(url, json=payload_happy)
    print(json.dumps(r1.json(), indent=2))
    
    print("\n--- SENDING ANOMALOUS FRAUD PAYLOAD ---")
    r2 = client.post(url, json=payload_fraud)
    print(json.dumps(r2.json(), indent=2))

In [2]:
import httpx
import json

# Point directly to your central Saga Ingress Core Gateway port
URL = "http://127.0.0.1:8000/api/v1/adjudicate"

print("====================================================")
print("🚀 STARTING MEDICLAIM-AI DISTRIBUTED SAGA EMULATOR   ")
print("====================================================\n")

# ---------------------------------------------------------------------
# SCENARIO 1: The Happy Path (Low Risk Claim Evaluation)
# ---------------------------------------------------------------------
payload_happy = {
    "policy_holder_id": "POL_BITS_9901",
    "customer_age": 34,
    "policy_deductible": 1000,
    "claim_amount": 45000.0,
    "past_claims_count": 0,
    "incident_hour": 14  # Regular mid-day hour profile
}

# ---------------------------------------------------------------------
# SCENARIO 2: The Early Exit Path (High Anomaly Behavioral Risk)
# ---------------------------------------------------------------------
payload_anomalous = {
    "policy_holder_id": "POL_BITS_4412",
    "customer_age": 21,
    "policy_deductible": 500,
    "claim_amount": 145000.0,
    "past_claims_count": 4,
    "incident_hour": 2  # Deep night hour + extreme history + high cost triggers classifier
}

with httpx.Client() as client:
    # --- Execute Scenario 1 ---
    print("▶ Sending Happy Path payload to cluster gateway...")
    try:
        response_1 = client.post(URL, json=payload_happy, timeout=5.0)
        print(f"Status Code: {response_1.status_code}")
        print("Response Payload JSON:")
        print(json.dumps(response_1.json(), indent=2))
    except Exception as e:
        print(f"💥 Network connection failed on transaction: {str(e)}")
        print("Check if your main.py server on Port 8000 is fully running.")

    print("\n" + "-"*50 + "\n")

    # --- Execute Scenario 2 ---
    print("▶ Sending Anomalous Fraud Profile payload to cluster gateway...")
    try:
        response_2 = client.post(URL, json=payload_anomalous, timeout=5.0)
        print(f"Status Code: {response_2.status_code}")
        print("Response Payload JSON:")
        print(json.dumps(response_2.json(), indent=2))
    except Exception as e:
        print(f"💥 Network connection failed on transaction: {str(e)}")

print("\n====================================================")
print("🏁 END OF VALIDATION RUN - SCREENSHOT TERMINAL NOW  ")
print("====================================================")

🚀 STARTING MEDICLAIM-AI DISTRIBUTED SAGA EMULATOR   

▶ Sending Happy Path payload to cluster gateway...
Status Code: 200
Response Payload JSON:
{
  "transaction_id": "1eb016c7-424f-4652-b936-7162cd105e56",
  "fraud_risk_score": 0.10122359544038773,
  "status": "CLAIM_APPROVED_SUCCESSFULLY",
  "approved_room_rent_allocation": 4500.0,
  "disclaimer": "Disclaimer: This calculation is an algorithmic reference evaluation under IRDAI guidelines and does not replace human clinical auditor authorization."
}

--------------------------------------------------

▶ Sending Anomalous Fraud Profile payload to cluster gateway...
Status Code: 200
Response Payload JSON:
{
  "transaction_id": "1b0bf913-54c6-49fc-b463-580471c13e9a",
  "fraud_risk_score": 0.9488250613212585,
  "status": "REJECTED_AUDIT_FLAGGED",
  "approved_room_rent_allocation": 0.0,
  "disclaimer": "Suspended due to behavioral fraud risk flags."
}

🏁 END OF VALIDATION RUN - SCREENSHOT TERMINAL NOW  
